---
title: "11. Porting jobs & apps to ACA"
description: "Map the Compose feature environment onto Azure Container Apps while reusing the same workload images, entrypoints, contracts, and behavioral checks."
categories: []
---

The local Compose stack is the runnable reference implementation for this course. Porting it to Azure Container Apps (ACA) means replacing local adapters with managed identities, managed storage, ingress, schedules, and control-plane triggers. The workload source does not fork: the same images and entrypoints run in both phases.


## One workload, two adapters

The mapping is explicit in the Terraform modules:

| Local Compose | ACA definition | Shared workload |
|---|---|---|
| train service | modules/train_job manual/scheduled ACA Job | src/train_job/Dockerfile → train.py |
| batch service | modules/batch_job manual/scheduled ACA Job | src/batch_job/Dockerfile → score.py |
| serving service | modules/serving_app ACA App | src/serving_app/Dockerfile → FastAPI app |
| dashboard service | modules/dashboard ACA App | src/dashboard/Dockerfile → dashboard API |
| llm Compose profile Jobs | modules/llm_job manual ACA Jobs | train image → register_llm.py or ml_platform.llm.evaluator |

The local runner is a small execution-plane adapter for train and batch. It
validates scalar parameters, starts subprocesses, and gives the dashboard an
asynchronous HTTP surface. The LLM profile uses the actual train workload image
directly, while ACA supplies the production execution plane. The control-plane
mechanisms can differ; the jobs still write the same results rows and use the
same model registry contracts.


## Configuration is where the environments differ

Compose answers shared questions with service names and demo credentials:

~~~text
MLFLOW_TRACKING_URI=http://mlflow:5000
PGHOST=postgres
PGUSER=mlplatform
PGPASSWORD=demo-password
~~~

ACA answers the same questions with managed-identity environment variables:

~~~text
MLFLOW_TRACKING_URI=https://<mlflow-app>/...
PGHOST=<postgres-fqdn>
PGUSER=id-jobs-train or id-jobs-batch
PGSSLMODE=require
AZURE_CLIENT_ID=<workload-identity-client-id>
~~~

The image stays the same. `DefaultAzureCredential` obtains tokens in Azure, while the results store uses password mode when Compose explicitly supplies `PGPASSWORD`. Object storage and Key Vault access follow the same adapter boundary: local MinIO/demo values are replaced by Azure role assignments and identity-backed clients.


## Schedules must preserve effective arguments

A schedule is not equivalent merely because ACA accepted a cron expression. It must launch the same workload with the same effective arguments.

The batch module therefore injects `DATA_SOURCE` and `MODEL_NAME` into the Job template. `score.py` treats `DATA_SOURCE` as the environment equivalent of `--data-source`, and defaults the model reference to the shared `production` alias when no exact version is supplied. The local Compose runner passes the corresponding flags explicitly, which makes ad-hoc runs easy to inspect.

For LLM workflows, the `llm_job` adapter uses the train image but changes only the container command. Registration runs `python register_llm.py`; evaluation runs `python -m ml_platform.llm.evaluator` with `LLM_EVAL_DATASET`, `LLM_MODEL_NAME`, and `LLM_MODEL_VERSION`. Local evaluation uses the bundled JSONL fixture and `MODEL_API_KEY`; cloud evaluation resolves the API credential through Key Vault and the train identity.


## Apps, identity, and promotion

The serving module configures the same `/healthz`, `/readyz`, and `/v1/predictions` endpoints that Compose exposes. ACA adds an HTTPS ingress, a user-assigned `id-serving` identity, and HTTP probes. The dashboard adds an HTTPS ingress, Easy Auth at the platform edge, a scoped Job-start role, and the `id-dashboard` results-reader identity.

Promotion still has two shared facts: the MLflow `production` alias points to version N, and the serving consumer is pinned to `MODEL_VERSION=N`. `demo/promote.py` accepts `--tracking-uri` so local and cloud registries are selected by configuration; its local and ACA branches only differ in how the serving consumer is repinned.

The Azure smoke adapter starts train and batch executions through `az`, polls each execution to a terminal `Succeeded`/failure state, checks the results API for a successful batch parent, and verifies serving readiness plus a prediction response. The local `demo/golden_path.py` performs the same behavioral assertions through the local runner. The trigger mechanisms remain separate, but the evidence they demand is shared.


## Deploy and inspect

The full deployment is intentionally staged so a missing image cannot create a half-configured workload:

~~~bash
cd projects/ml-platform
./deploy/deploy.sh --pg-admin-upn you@example.com
./deploy/smoke-tests.sh --tf-vars infra/environments/dev.tfvars
~~~

PowerShell users can use `deploy.ps1` and `smoke-tests.ps1`. Set `LLM_EVAL_DATASET` when the cloud evaluation Job should be provisioned; the registration Job is enabled whenever the shared train image is deployed.

The deployment adapter is allowed to know about Azure resource names, managed identities, ACR, ACA schedules, and `az` APIs. It is not allowed to invent a second training implementation, model loader, results schema, or readiness contract.

Next: [12 — CI/CD](12-ci-cd.ipynb) makes the digest and validation path repeatable.
